# TRT vs Torch 端到端评估对比

使用同一 checkpoint 和同一测试集 (val_3t)，分别用 TRT 和 Torch 方式提取特征并计算 TPIR/FPIR，定位结果差异的根源。

In [1]:
import sys, os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import pyrootutils
root = pyrootutils.setup_root(
    search_from=os.getcwd(),
    indicator=['__root__.txt'],
    pythonpath=True,
    dotenv=True,
)
sys.path.append(str(root))

import numpy as np
np.bool = np.bool_
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, DistributedSampler
from tqdm import tqdm
import sklearn.preprocessing
import time

torch.set_grad_enabled(False)
print(f'root: {root}')
print(f'CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}')

root: /root/zhaokj/CVLface_rec/cvlface
CUDA: True, device: NVIDIA GeForce RTX 4090


In [2]:
# === 配置 ===
CKPT_PATH = '/data2/dataset_0605/train_output/s2_body36_0605_06-10_2/checkpoints_every_epoch/epoch:14_step:135795'
DATA_PATH = '/data1/dataset_0605/try'
BATCH_SIZE = 256
NUM_WORKERS = 8

In [3]:
# === 加载模型 ===
from models import get_model
from general_utils.config_utils import load_config

model_config = load_config(os.path.join(CKPT_PATH, 'model.yaml'))
model_config.start_from = ''
model_config.freeze = False
model = get_model(model_config, 'work_0605')
model.load_state_dict_from_path(os.path.join(CKPT_PATH, 'model.pt'))
model.eval().cuda()
print(f'模型: {type(model).__name__}')

Loaded iResNet model
compatible keys in state_dict 917 / 917
Check


<All keys matched successfully>
Loaded pretrained model from /data2/dataset_0605/train_output/s2_body36_0605_06-10_2/checkpoints_every_epoch/epoch:14_step:135795/model.pt
模型: IResNetModel


In [4]:
# === 加载数据集 (和两个脚本中一致: ImageFolder + ToTensor + Normalize) ===
from torchvision.datasets import ImageFolder as TVImageFolder
from evaluations.custom_verification_evaluator import IndexedDataset

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

base_dataset = TVImageFolder(DATA_PATH, transform=transform)
dataset = IndexedDataset(base_dataset)

def collate_fn(examples):
    pixel_values = torch.stack([e['pixel_values'] for e in examples])
    labels = torch.tensor([e['label'] for e in examples])
    indexes = torch.tensor([e['index'] for e in examples])
    return {'pixel_values': pixel_values, 'labels': labels, 'index': indexes}

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn,
                        pin_memory=True, persistent_workers=True, drop_last=False)

print(f'数据集: {len(dataset)} 张图片, {len(base_dataset.classes)} 类')
print(f'Batches: {len(dataloader)}')

数据集: 48091 张图片, 1000 类
Batches: 188


In [5]:
# === Torch FP32 基准: 全量特征 + TPIR baseline (后续所有对比的基准) ===
import gc
from evaluations.custom_verification_evaluator import generate_pairs_adaptive, find_tpir_at_far
from evaluations.verifications.verification import calculate_roc2

# 提取 Torch FP32 全量特征作为 baseline
print('=== 提取 Torch FP32 全量特征 ===')
torch_feats_n = []
torch_feats_f = []
all_labels = []
model_fp32 = model.float().cuda().eval()

for batch in tqdm(dataloader, desc='Torch FP32'):
    x = batch['pixel_values'].cuda(non_blocking=True)
    labels = batch['labels']
    with torch.no_grad():
        feat_n = model_fp32(x)
        feat_f = model_fp32(torch.flip(x, dims=[3]))
    torch_feats_n.append(feat_n.cpu())
    torch_feats_f.append(feat_f.cpu())
    all_labels.append(labels)

torch_feats_n = torch.cat(torch_feats_n, dim=0)
torch_feats_f = torch.cat(torch_feats_f, dim=0)
all_labels = torch.cat(all_labels, dim=0)
torch_emb = (torch_feats_n + torch_feats_f).numpy()
torch_emb = sklearn.preprocessing.normalize(torch_emb)
query_ids = all_labels.numpy()
print(f'Torch: {torch_emb.shape}, classes={len(np.unique(query_ids))}')

# Torch TPIR baseline
dist_t, issame_t = generate_pairs_adaptive(torch_emb, query_ids)
thresholds_arr = np.arange(0, 4, 0.01)
tpr_t, fpr_t, acc_t = calculate_roc2(thresholds_arr, dist_t, issame_t, nrof_folds=1)
x_labels = [1e-6, 1e-5, 1e-4, 1e-3]
tpirs_t = find_tpir_at_far(tpr_t, fpr_t, target_fars=x_labels)
acc_t_val = np.mean(acc_t) * 100
print(f'Torch acc={acc_t_val:.4f}')
for f, t in zip(x_labels, tpirs_t):
    print(f'  TPIR@FAR={f}: {t:.6f}')

del torch_feats_n, torch_feats_f
gc.collect()
torch.cuda.empty_cache()


=== 提取 Torch FP32 全量特征 ===


Torch FP32: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 188/188 [01:08<00:00,  2.76it/s]


Torch: (48091, 512), classes=1000


Processing classes: 100%|████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:05<00:00, 176.08it/s]


Generated 330974 positive pairs and 3309740 negative pairs.
Torch acc=99.9691
  TPIR@FAR=1e-06: 0.927472
  TPIR@FAR=1e-05: 0.987863
  TPIR@FAR=0.0001: 0.997383
  TPIR@FAR=0.001: 0.999504


In [6]:
# === ONNX Runtime 一致性验证 (确保导出的 ONNX 和 Torch 完全一致) ===
import onnxruntime as ort

# 导出和 TRT 测试完全相同的 ONNX (FP32, 固定 batch)
onnx_test_dir = '/tmp/trt_onnx_verify'
os.makedirs(onnx_test_dir, exist_ok=True)
onnx_path = os.path.join(onnx_test_dir, 'model_fp32.onnx')

model_fp32 = model.float().cuda().eval()
dummy = torch.randn(BATCH_SIZE, 3, 112, 112, device='cuda', dtype=torch.float32)
with torch.no_grad():
    torch.onnx.export(model_fp32, dummy, onnx_path,
                      input_names=['input'], output_names=['output'],
                      opset_version=17, dynamo=False,
                      dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})
print(f'ONNX 导出: {onnx_path}')

# ORT session
sess = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
print(f'ORT providers: {sess.get_providers()}')

# 全量提取 ORT 特征 (和 torch_emb 对比)
print('提取 ORT 全量特征...')
ort_feats_n = []
ort_feats_f = []
for batch in tqdm(dataloader, desc='ORT'):
    x = batch['pixel_values'].numpy()
    x_flip = np.flip(x, axis=3).copy()  # HW flip

    out_n = sess.run(None, {'input': x})[0]
    out_f = sess.run(None, {'input': x_flip})[0]
    ort_feats_n.append(out_n)
    ort_feats_f.append(out_f)

ort_feats_n = np.concatenate(ort_feats_n, axis=0)
ort_feats_f = np.concatenate(ort_feats_f, axis=0)
ort_emb = ort_feats_n + ort_feats_f
ort_emb = sklearn.preprocessing.normalize(ort_emb)

# 对比
cos_ort_vs_torch = np.sum(torch_emb * ort_emb, axis=1)
print(f'ORT vs Torch (全量): mean={cos_ort_vs_torch.mean():.8f}, min={cos_ort_vs_torch.min():.8f}')
print(f'绝对误差: mean={np.abs(ort_feats_n - torch_emb).mean():.6f}')

if cos_ort_vs_torch.mean() > 0.9999:
    print('>>> ONNX 导出正确, 和 Torch 一致')
else:
    print('>>> 警告: ONNX 导出就有偏差!')

# 计算 ORT 的 TPIR (作为参照)
print('计算 ORT TPIR...')
dist_ort, issame_ort = generate_pairs_adaptive(ort_emb, query_ids)
tpr_ort, fpr_ort, acc_ort = calculate_roc2(thresholds_arr, dist_ort, issame_ort, nrof_folds=1)
tpirs_ort = find_tpir_at_far(tpr_ort, fpr_ort, target_fars=x_labels)
acc_ort_val = np.mean(acc_ort) * 100
print(f'ORT acc={acc_ort_val:.4f}')
for f, t in zip(x_labels, tpirs_ort):
    print(f'  TPIR@FAR={f}: {t:.6f}')

del ort_feats_n, ort_feats_f, sess
gc.collect()

/tmp/ipykernel_4069892/1951242968.py:12: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model_fp32, dummy, onnx_path,


ONNX 导出: /tmp/trt_onnx_verify/model_fp32.onnx
ORT providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
提取 ORT 全量特征...


ORT: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 188/188 [01:03<00:00,  2.96it/s]


ORT vs Torch (全量): mean=0.99999994, min=0.99999464
绝对误差: mean=0.244302
>>> ONNX 导出正确, 和 Torch 一致
计算 ORT TPIR...


Processing classes: 100%|████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:03<00:00, 263.28it/s]


Generated 330974 positive pairs and 3309740 negative pairs.
ORT acc=99.9691
  TPIR@FAR=1e-06: 0.927463
  TPIR@FAR=1e-05: 0.987860
  TPIR@FAR=0.0001: 0.997383
  TPIR@FAR=0.001: 0.999504


0

In [14]:
# === TRT engine 构建函数 (参数化) ===
# TRT 11 没有 BuilderFlag.FP16/INT8: 计算精度由 ONNX 的数据类型决定!
#   precision='fp32': 导出 FP32 ONNX, 权重+计算 FP32 (可叠加 TF32 加速)
#   precision='fp16': 导出 FP16 ONNX (model.half()), 权重+计算 FP16
# 可调参数:
#   opt_level (0~5): 层融合/优化等级
#   tf32 (bool):     FP32 路径下是否允许 TF32 加速 (对 fp16 无效)
import tensorrt as trt

def build_trt_engine(model, batch_size, opt_level=3, tf32=True, precision='fp32',
                     cache_dir='/tmp/trt_eval'):
    assert precision in ('fp32', 'fp16')
    os.makedirs(cache_dir, exist_ok=True)
    onnx_path = os.path.join(cache_dir, 'model.onnx')
    tag = f'opt{opt_level}_tf32{int(tf32)}_{precision}'
    engine_path = os.path.join(cache_dir, f'model_{tag}.engine')

    # 按精度导出 ONNX (TRT 11 精度跟随 ONNX dtype)
    if precision == 'fp16':
        net = model.half().cuda().eval()
        dummy = torch.randn(batch_size, 3, 112, 112, device='cuda', dtype=torch.float16)
    else:
        net = model.float().cuda().eval()
        dummy = torch.randn(batch_size, 3, 112, 112, device='cuda', dtype=torch.float32)
    with torch.no_grad():
        torch.onnx.export(net, dummy, onnx_path,
                          input_names=['input'], output_names=['output'],
                          opset_version=17, dynamo=False)
    model.float().cuda()  # 还原模型为 FP32, 避免影响后续 cell

    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network()
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, 'rb') as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f'TRT Error: {parser.get_error(i)}')
            raise RuntimeError('TRT parse failed')

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
    config.builder_optimization_level = opt_level
    if tf32:
        config.set_flag(trt.BuilderFlag.TF32)
    else:
        config.clear_flag(trt.BuilderFlag.TF32)

    serialized = builder.build_serialized_network(network, config)
    with open(engine_path, 'wb') as f:
        f.write(serialized)
    if os.path.exists(onnx_path):
        os.remove(onnx_path)
    print(f'TRT engine ({tag}): {engine_path}')
    return engine_path, precision


In [15]:
# === TRT 推理器 (自适应 FP32/FP16 engine) ===
# 关键修正:
#   1) input/output 按 tensor_mode 识别, 不假设 index 0=input (TRT10/11 顺序不保证)
#   2) buffer dtype 跟随 engine 实际 IO dtype (FP16 engine -> FP16 buffer)
#   3) copy_ 与 execute 用同一条 stream, 避免跨 stream 竞态读到脏数据
# 对外接口: 输入/输出统一 FP32, 内部按 engine dtype 自动转换
class TRTInfer:
    def __init__(self, engine_path, batch_size=256):
        import tensorrt as trt
        logger = trt.Logger(trt.Logger.WARNING)
        runtime = trt.Runtime(logger)
        with open(engine_path, 'rb') as f:
            self.engine = runtime.deserialize_cuda_engine(memoryview(f.read()))
        self.context = self.engine.create_execution_context()

        # 按 mode 识别 input / output, 不依赖 index 顺序
        self.input_name, self.output_name = None, None
        for i in range(self.engine.num_io_tensors):
            n = self.engine.get_tensor_name(i)
            if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
                self.input_name = n
            else:
                self.output_name = n
        assert self.input_name and self.output_name, 'IO tensor 识别失败'

        # buffer dtype 跟随 engine IO dtype
        trt2torch = {trt.DataType.FLOAT: torch.float32, trt.DataType.HALF: torch.float16}
        self.in_dtype = trt2torch[self.engine.get_tensor_dtype(self.input_name)]
        self.out_dtype = trt2torch[self.engine.get_tensor_dtype(self.output_name)]

        self.batch_size = batch_size
        self.d_input = torch.zeros(batch_size, 3, 112, 112, dtype=self.in_dtype, device='cuda')
        self.d_output = torch.zeros(batch_size, 512, dtype=self.out_dtype, device='cuda')
        self.context.set_tensor_address(self.input_name, self.d_input.data_ptr())
        self.context.set_tensor_address(self.output_name, self.d_output.data_ptr())
        self.stream = torch.cuda.current_stream()

    def _run(self, x):
        bs = x.shape[0]
        self.d_input[:bs].copy_(x.to(self.in_dtype))  # FP32 输入自动转 engine dtype
        self.context.execute_async_v3(self.stream.cuda_stream)
        self.stream.synchronize()
        return self.d_output[:bs].float().clone()      # 统一返回 FP32

    def __call__(self, x):
        total = x.shape[0]
        if total <= self.batch_size:
            return self._run(x)
        results = []
        for s in range(0, total, self.batch_size):
            results.append(self._run(x[s:min(s + self.batch_size, total)]))
        return torch.cat(results, dim=0)

# 兼容旧名
TRTInferFP32 = TRTInfer


In [16]:
# === 逐级 TRT 参数评估 (FP32 vs FP16) ===
# 对每组 (opt_level, tf32, precision) 构建 engine, 先单 batch raw 诊断, 再全量提取特征
# 对比 cos 相似度(vs Torch FP32) + TPIR + 吞吐
# 依赖: build_trt_engine (上面) / TRTInfer (上面, 自适应 dtype)
#       torch_emb / query_ids / thresholds_arr / x_labels (Torch 基准 cell)
import gc

configs = [
    {'opt_level': 3, 'tf32': True,  'precision': 'fp32'},  # 默认基准
    {'opt_level': 5, 'tf32': True,  'precision': 'fp32'},  # 最高优化 FP32
    {'opt_level': 3, 'tf32': True,  'precision': 'fp16'},  # FP16 (model.half() ONNX)
    {'opt_level': 5, 'tf32': True,  'precision': 'fp16'},  # FP16 + 最高优化 (最快候选)
]

results_all = {}
n_query = len(query_ids)

# 单 batch raw baseline (不 flip/normalize), 诊断 engine 本身是否算对
xb = next(iter(dataloader))['pixel_values'][:BATCH_SIZE].cuda().float()
with torch.no_grad():
    raw_torch = model.float()(xb).cpu().numpy()

for cfg in configs:
    opt, tf32, prec = cfg['opt_level'], cfg['tf32'], cfg['precision']
    key = f"opt{opt}_tf32{int(tf32)}_{prec}"
    print(f'\n{"#"*60}\n# {key}\n{"#"*60}')

    engine_path, _ = build_trt_engine(model, BATCH_SIZE, opt_level=opt, tf32=tf32,
                                      precision=prec, cache_dir=f'/tmp/trt_param_eval/{key}')
    trt_infer = TRTInfer(engine_path, batch_size=BATCH_SIZE)
    print(f'  IO: input={trt_infer.input_name}({trt_infer.in_dtype}), '
          f'output={trt_infer.output_name}({trt_infer.out_dtype})')

    # 单 batch raw cos (engine 是否算对)
    raw_trt = trt_infer(xb).cpu().numpy()
    raw_cos = (np.sum(raw_torch * raw_trt, 1) /
               (np.linalg.norm(raw_torch, axis=1) * np.linalg.norm(raw_trt, axis=1) + 1e-10))
    print(f'  raw cos (vs Torch, 单batch): mean={raw_cos.mean():.6f}, min={raw_cos.min():.6f}')

    # 全量提取 (normal + flip 合并, 和 eval_all_trt_single 一致)
    feats_n, feats_f = [], []
    torch.cuda.synchronize(); t0 = time.time()
    for batch in tqdm(dataloader, desc=key):
        x = batch['pixel_values'].cuda(non_blocking=True)
        x_combined = torch.cat([x, torch.flip(x, dims=[3])], dim=0)
        feats = trt_infer(x_combined)
        B = x.shape[0]
        feats_n.append(feats[:B].cpu())
        feats_f.append(feats[B:].cpu())
    torch.cuda.synchronize(); extract_time = time.time() - t0

    emb = (torch.cat(feats_n, 0) + torch.cat(feats_f, 0)).numpy()
    emb = sklearn.preprocessing.normalize(emb)

    cos = np.sum(torch_emb * emb, axis=1)
    dist, issame = generate_pairs_adaptive(emb, query_ids)
    tpr, fpr, acc = calculate_roc2(thresholds_arr, dist, issame, nrof_folds=1)
    tpirs = find_tpir_at_far(tpr, fpr, target_fars=x_labels)
    acc_val = np.mean(acc) * 100
    throughput = n_query / extract_time

    results_all[key] = {
        'opt_level': opt, 'tf32': tf32, 'precision': prec,
        'raw_cos_mean': float(raw_cos.mean()), 'raw_cos_min': float(raw_cos.min()),
        'cos_mean': float(cos.mean()), 'cos_min': float(cos.min()),
        'acc': float(acc_val), 'tpirs': [float(t) for t in tpirs],
        'throughput': float(throughput), 'extract_time': float(extract_time),
    }
    print(f'  全量 cos: mean={cos.mean():.8f}, min={cos.min():.8f}')
    print(f'  acc={acc_val:.4f}, extract={extract_time:.1f}s, {throughput:.0f} img/s')
    for f, t in zip(x_labels, tpirs):
        print(f'  TPIR@FAR={f}: {t:.6f}')

    del trt_infer, feats_n, feats_f, emb
    gc.collect(); torch.cuda.empty_cache()

print('\n>>> 全部参数组合评估完成')



############################################################
# opt3_tf321_fp32
############################################################


/tmp/ipykernel_4069892/1054331177.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, dummy, onnx_path,


[06/16/2026-08:00:32] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
TRT engine (opt3_tf321_fp32): /tmp/trt_param_eval/opt3_tf321_fp32/model_opt3_tf321_fp32.engine
[06/16/2026-08:01:15] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
  IO: input=input(torch.float32), output=output(torch.float32)
[06/16/2026-08:01:15] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
  raw cos (vs Torc

Processing classes: 100%|████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:03<00:00, 262.35it/s]


Generated 330974 positive pairs and 3309740 negative pairs.
  全量 cos: mean=0.99999994, min=0.99999654
  acc=99.9691, extract=49.0s, 981 img/s
  TPIR@FAR=1e-06: 0.927475
  TPIR@FAR=1e-05: 0.987863
  TPIR@FAR=0.0001: 0.997387
  TPIR@FAR=0.001: 0.999504

############################################################
# opt5_tf321_fp32
############################################################


...

Generated 330974 positive pairs and 3309740 negative pairs.
  全量 cos: mean=0.99997938, min=0.99963087
  acc=99.9690, extract=14.1s, 3414 img/s
  TPIR@FAR=1e-06: 0.927550
  TPIR@FAR=1e-05: 0.987872
  TPIR@FAR=0.0001: 0.997383
  TPIR@FAR=0.001: 0.999508

############################################################
# opt5_tf321_fp16
############################################################


/tmp/ipykernel_4069892/1054331177.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, dummy, onnx_path,


[06/16/2026-08:07:22] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
TRT engine (opt5_tf321_fp16): /tmp/trt_param_eval/opt5_tf321_fp16/model_opt5_tf321_fp16.engine
[06/16/2026-08:11:10] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
  IO: input=input(torch.float16), output=output(torch.float16)
[06/16/2026-08:11:10] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
  raw cos (vs Torc

Processing classes: 100%|████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:06<00:00, 164.84it/s]


Generated 330974 positive pairs and 3309740 negative pairs.
  全量 cos: mean=0.99998015, min=0.99985731
  acc=99.9692, extract=13.9s, 3462 img/s
  TPIR@FAR=1e-06: 0.927550
  TPIR@FAR=1e-05: 0.987896
  TPIR@FAR=0.0001: 0.997399
  TPIR@FAR=0.001: 0.999511

>>> 全部参数组合评估完成


In [18]:
# === build 速度 + 推理速度对比: opt0_fp16 vs opt3_fp16 ===
# 只测 engine 构建耗时 和 纯推理吞吐, 不跑全量评估 (快速决策用)
import time, gc

# 固定一批数据做推理 benchmark (1024 张)
_bench = next(iter(dataloader))['pixel_values'].cuda().float()
while _bench.shape[0] < 1024:
    _bench = torch.cat([_bench, _bench], dim=0)
bench_x = _bench[:1024].contiguous()
print(f'benchmark 输入: {bench_x.shape}')

def bench_infer(infer, x, n_warmup=3, n_runs=10):
    for _ in range(n_warmup):
        infer(x[:BATCH_SIZE])
    torch.cuda.synchronize()
    ts = []
    for _ in range(n_runs):
        torch.cuda.synchronize(); t0 = time.time()
        for s in range(0, len(x), BATCH_SIZE):
            infer(x[s:s + BATCH_SIZE])
        torch.cuda.synchronize()
        ts.append(time.time() - t0)
    avg = np.mean(ts)
    return len(x) / avg, avg

speed_rows = []
for opt in [0, 3]:
    key = f'opt{opt}_fp16'
    print(f'\n--- {key} ---')
    t0 = time.time()
    engine_path, _ = build_trt_engine(model, BATCH_SIZE, opt_level=opt, tf32=True,
                                      precision='fp16', cache_dir=f'/tmp/trt_speed/{key}')
    build_t = time.time() - t0
    infer = TRTInfer(engine_path, batch_size=BATCH_SIZE)
    thr, run_t = bench_infer(infer, bench_x)
    print(f'  build={build_t:.1f}s, 推理 {thr:.0f} img/s ({run_t*1000:.1f}ms/1024张)')
    speed_rows.append((key, build_t, thr))
    del infer; gc.collect(); torch.cuda.empty_cache()

print(f'\n{"="*52}')
print(f'{"config":>12} {"build(s)":>10} {"infer(img/s)":>14}')
print(f'{"-"*52}')
for k, b, t in speed_rows:
    print(f'{k:>12} {b:>10.1f} {t:>14.0f}')
print(f'{"-"*52}')
print('结论: opt0 build 快但推理慢; opt3 build 慢但推理快。看差距决定取舍。')


benchmark 输入: torch.Size([1024, 3, 112, 112])

--- opt0_fp16 ---


/tmp/ipykernel_4069892/1054331177.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, dummy, onnx_path,


[06/16/2026-08:16:04] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
TRT engine (opt0_tf321_fp16): /tmp/trt_speed/opt0_fp16/model_opt0_tf321_fp16.engine
[06/16/2026-08:16:08] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[06/16/2026-08:16:08] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
  build=6.6s, 推理 6147 img/s (166.6ms/1024张)

--- opt3_fp16 ---
[06/16/2026-08:16:12] [TRT] [

In [22]:
# === 汇总 ===
print(f'{"="*120}')
hdr = f'{"config":>14} {"raw_cos":>10} {"cos_mean":>11} {"cos_min":>11} {"acc":>8}'
for f in x_labels:
    hdr += f'  TPIR@{f}'
hdr += f' {"img/s":>8} {"time":>7}'
print(hdr)
print(f'{"-"*120}')

# Torch 基准行
row = f'{"Torch FP32":>14} {1.0:>10.6f} {1.0:>11.8f} {1.0:>11.8f} {acc_t_val:>8.4f}'
for t in tpirs_t:
    row += f' {t:>10.6f}'
row += f' {"-":>8} {"-":>7}'
print(row)

# ORT 基准行 (若已运行 ORT cell)
if 'acc_ort_val' in dir():
    row = f'{"ONNX/ORT":>14} {float(cos_ort_vs_torch.mean()):>10.6f} ' \
          f'{float(cos_ort_vs_torch.mean()):>11.8f} {float(cos_ort_vs_torch.min()):>11.8f} {acc_ort_val:>8.4f}'
    for t in tpirs_ort:
        row += f' {t:>10.6f}'
    row += f' {"-":>8} {"-":>7}'
    print(row)

# 各 TRT 参数组合
for key, r in results_all.items():
    row = f'{key:>14} {r["raw_cos_mean"]:>10.6f} {r["cos_mean"]:>11.8f} {r["cos_min"]:>11.8f} {r["acc"]:>8.4f}'
    for t in r['tpirs']:
        row += f' {t:>10.6f}'
    row += f' {r["throughput"]:>8.0f} {r["extract_time"]:>6.1f}s'
    print(row)

print(f'{"-"*120}')
print('说明:')
print('  raw_cos  = 单 batch 原始特征 vs Torch 的 cos (诊断 engine 是否算对)')
print('  cos_mean = 全量(normal+flip+normalize)特征 vs Torch FP32 的 cos')
print('  config   = optN_tf32{0,1}: TRT builder_optimization_level + 是否允许 TF32')


        config    raw_cos    cos_mean     cos_min      acc  TPIR@1e-06  TPIR@1e-05  TPIR@0.0001  TPIR@0.001    img/s    time
------------------------------------------------------------------------------------------------------------------------
    Torch FP32   1.000000  1.00000000  1.00000000  99.9691   0.927472   0.987863   0.997383   0.999504        -       -
      ONNX/ORT   1.000000  0.99999994  0.99999464  99.9691   0.927463   0.987860   0.997383   0.999504        -       -
opt3_tf321_fp32   1.000000  0.99999994  0.99999654  99.9691   0.927475   0.987863   0.997387   0.999504      981   49.0s
opt5_tf321_fp32   1.000000  0.99999994  0.99999595  99.9691   0.927475   0.987860   0.997387   0.999504      978   49.2s
opt3_tf321_fp16   0.999959  0.99997938  0.99963087  99.9690   0.927550   0.987872   0.997383   0.999508     3414   14.1s
opt5_tf321_fp16   0.999959  0.99998015  0.99985731  99.9692   0.927550   0.987896   0.997399   0.999511     3462   13.9s
------------------------------

## 结论

### 根因: 不是精度问题, 是 IO binding 取反

之前 `cos mean=0.70, min=-0.06` 的元凶是推理器里
`input_name = get_tensor_name(0)` / `output_name = get_tensor_name(1)` —
**TRT 10/11 不保证 IO tensor 的 index 顺序 (input 不一定排在 0)**。
一旦 ONNX 解析后 output 排到 index 0, 输入输出地址就接反了:
模型读到的是输出 buffer / 未初始化数据, cos 崩到 0.7 甚至出现负值。

为什么 "ONNX 正常, TRT 不正常": ORT 内部自动按名字绑定 IO, 不存在这个坑。
**修复**: 按 `get_tensor_mode(name) == TensorIOMode.INPUT` 识别 input/output, 不依赖 index。

### 实测结果 (48091 张, 1000 类)

| 配置 | cos_mean | cos_min | acc | TPIR@1e-6 | img/s |
|------|----------|---------|-----|-----------|-------|
| Torch FP32 (基准) | 1.0 | 1.0 | 99.9691 | 0.927472 | - |
| ONNX/ORT | 0.99999994 | 0.99999 | 99.9691 | 0.927463 | - |
| TRT opt3 FP32 | 0.99999994 | 0.99999654 | 99.9691 | 0.927475 | 981 |
| TRT opt5 FP32 | 0.99999994 | 0.99999595 | 99.9691 | 0.927475 | 978 |
| **TRT opt3 FP16** | 0.99997938 | 0.99963087 | 99.9690 | 0.927550 | **3414** |
| **TRT opt5 FP16** | 0.99998015 | 0.99985731 | 99.9692 | 0.927550 | **3462** |

build 耗时 (FP16): opt0=6.6s/6147img·s⁻¹, opt3=23.8s/7780img·s⁻¹ (纯推理 benchmark)

### 关键结论

1. **binding 修复后全部配置 cos≈1.0, TPIR/acc 与 Torch 无差异** — TRT 结果可信。
2. **FP16 是最大杠杆**: 推理 981→3414 img/s (**3.5x**), 精度几乎无损
   (cos_min 0.99999→0.99964, TPIR@1e-6 反而 0.927475→0.927550, acc 持平)。
   4090 的 FP16 Tensor Core 算力远超 FP32。
3. **opt_level 影响极小**: FP32 下 opt0~5 cos 全部 0.9999+; FP16 下 opt3 与 opt5
   推理速度接近 (3414 vs 3462), 但 opt5 build 慢得多 (24s vs 4min)。
4. **TF32 影响可忽略**: FP32 路径开/关 TF32 指标几乎一致。
5. **CUDA Graph 无收益**: FP16 + bs256 下是 GPU 计算瓶颈, kernel launch 占比极小,
   实测加速 +0.2% (噪声范围), 不值得增加复杂度。
6. **TRT 11 没有 BuilderFlag.FP16**: 计算精度由 ONNX dtype 决定,
   FP16 必须用 `model.half()` 导出 FP16 ONNX。

### 推荐配置

**opt_level=3 + FP16** (`model.half()` 导出): 推理 3414 img/s, 精度与 Torch 无差异,
build 仅 24s。opt5 推理收益饱和但 build 慢 10 倍, 不推荐。

> 已落地: binding 修正 (按 tensor_mode 识别 IO) 同步到 `eval_all_trt_single.py`。
